# OpenVLA — Step 1: Data Preparation

This notebook downloads BridgeData V2 from HuggingFace, generates 7-DoF actions via optical flow,
saves episode folders, converts to HuggingFace DatasetDict format, validates, and uploads to S3.

**Pipeline:**
1. Download BridgeData V2 → save episode folders (images, actions.npy, language.txt)
2. Validate intermediate dataset
3. Convert episode folders → HuggingFace DatasetDict (train/validation split)
4. Upload to S3

**Runtime:** ~30-45 min for 600 episodes

## 1. Environment Setup

In [1]:
!pip install -q datasets pyarrow opencv-python-headless Pillow numpy tqdm sagemaker boto3

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
aiobotocore 2.22.0 requires botocore<1.37.4,>=1.37.2, but you have botocore 1.42.92 which is incompatible.
amazon-sagemaker-jupyter-ai-q-developer 1.2.9 requires numpy<=2.0.1, but you have numpy 2.4.4 which is incompatible.
amazon-sagemaker-sql-magic 0.1.4 requires numpy<2, but you have numpy 2.4.4 which is incompatible.
autogluon-common 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.4 which is incompatible.
autogluon-core 1.5.0 requires numpy<2.4.0,>=1.25.0, but you ha

In [2]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass("Enter your Hugging Face token: ")
login(token=hf_token)

Enter your Hugging Face token:  ········


In [3]:
import os, json, random
from pathlib import Path
import boto3, cv2, numpy as np, sagemaker
from datasets import Dataset, DatasetDict, load_dataset
from PIL import Image
from tqdm import tqdm

## 2. Configuration

In [4]:
from sagemaker.core.helper import session_helper

sagemaker_session = session_helper.Session()
region = sagemaker_session.boto_region_name
account_id = boto3.client('sts').get_caller_identity()['Account']
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

print(f'Region: {region}')
print(f'Account: {account_id}')
print(f'Bucket: {bucket_name}')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region: us-east-1
Account: 783764584149
Bucket: sagemaker-us-east-1-783764584149


In [5]:
MAX_EPISODES = 600
ACTION_DIM = 7
RAW_DATASET_DIR = './bridge_synthetic_dataset'
HF_DATASET_DIR = './bridge_hf_synthetic'

if default_prefix:
    S3_PREFIX = f'{default_prefix}/openvla-finetuning/datasets/bridge_hf_synthetic'
else:
    S3_PREFIX = 'openvla-finetuning/datasets/bridge_hf_synthetic'

print(f'Episodes: {MAX_EPISODES}')
print(f'S3 destination: s3://{bucket_name}/{S3_PREFIX}')

Episodes: 600
S3 destination: s3://sagemaker-us-east-1-783764584149/openvla-finetuning/datasets/bridge_hf_synthetic


## 3. Download BridgeData & Generate Actions (Optical Flow)

In [6]:
def estimate_actions_from_optical_flow(frames, action_dim=7):
    actions = []
    for i in range(len(frames) - 1):
        img1 = np.array(frames[i].convert('RGB'))
        img2 = np.array(frames[i + 1].convert('RGB'))
        gray1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
        gray2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
        flow = cv2.calcOpticalFlowFarneback(
            gray1, gray2, None, pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2, flags=0)
        mean_flow = np.mean(flow, axis=(0, 1))
        action = np.zeros(action_dim, dtype=np.float32)
        action[0] = mean_flow[0] / 100.0
        action[1] = mean_flow[1] / 100.0
        actions.append(action)
    actions.append(np.zeros(action_dim, dtype=np.float32))
    return np.array(actions, dtype=np.float32)

In [7]:
print('=== Downloading BridgeData V2 Scripted Images ===')
ds = load_dataset('VyoJ/BridgeData-V2-Scripted-Images')
data = ds['train']
num_episodes = min(len(data), MAX_EPISODES)
print(f'Source: {len(data)} episodes, using {num_episodes}')

os.makedirs(RAW_DATASET_DIR, exist_ok=True)

for idx in tqdm(range(num_episodes), desc='Generating'):
    sample = data[idx]
    episode_dir = os.path.join(RAW_DATASET_DIR, f'episode_{idx:06d}')
    os.makedirs(os.path.join(episode_dir, 'images'), exist_ok=True)
    frames = [f for f in [sample['first_image'], sample['intermediate_image'], sample['frame_43_image']] if f]
    for t, img in enumerate(frames):
        img.save(os.path.join(episode_dir, 'images', f'{t:06d}.jpg'))
    instruction = 'move object to target'
    actions = estimate_actions_from_optical_flow(frames, ACTION_DIM)
    if actions is None or actions.size == 0:
        actions = np.zeros((len(frames), ACTION_DIM), dtype=np.float32)
    actions = np.nan_to_num(actions, nan=0.0, posinf=0.0, neginf=0.0)
    np.save(os.path.join(episode_dir, 'actions.npy'), actions)
    with open(os.path.join(episode_dir, 'language.txt'), 'w') as f:
        f.write(instruction)
    with open(os.path.join(episode_dir, 'metadata.json'), 'w') as f:
        json.dump({'length': len(frames), 'method': 'optical_flow'}, f)

print(f'Raw dataset saved to: {RAW_DATASET_DIR}')

=== Downloading BridgeData V2 Scripted Images ===


Resolving data files:   0%|          | 0/26407 [00:00<?, ?it/s]

Source: 8802 episodes, using 600


Generating: 100%|██████████| 600/600 [01:26<00:00,  6.94it/s]

Raw dataset saved to: ./bridge_synthetic_dataset


## 4. Validate Intermediate Dataset

In [8]:
episodes = sorted([d for d in os.listdir(RAW_DATASET_DIR)
                   if os.path.isdir(os.path.join(RAW_DATASET_DIR, d)) and d.startswith('episode_')])
valid_count, invalid_count = 0, 0
for ep_name in episodes:
    ep_path = os.path.join(RAW_DATASET_DIR, ep_name)
    try:
        actions = np.load(os.path.join(ep_path, 'actions.npy'), allow_pickle=True)
        has_images = len([f for f in os.listdir(os.path.join(ep_path, 'images')) if f.endswith(('.jpg','.png'))]) > 0
        has_lang = os.path.exists(os.path.join(ep_path, 'language.txt'))
        no_nan = not np.any(np.isnan(actions)) and not np.any(np.isinf(actions))
        if has_images and has_lang and no_nan and actions.shape[-1] == 7:
            valid_count += 1
        else:
            invalid_count += 1
    except:
        invalid_count += 1
print(f'Total: {len(episodes)}, Valid: {valid_count}, Invalid: {invalid_count}')

Total: 600, Valid: 600, Invalid: 0


## 5. Convert to HuggingFace DatasetDict

In [9]:
hf_data = []
for ep_name in tqdm(episodes, desc='Converting to HF'):
    ep_path = os.path.join(RAW_DATASET_DIR, ep_name)
    try:
        image_dir = os.path.join(ep_path, 'images')
        image_files = sorted([f for f in os.listdir(image_dir) if f.endswith(('.jpg','.png'))])
        if not image_files: continue
        imgs = [Image.open(os.path.join(image_dir, f)) for f in image_files]
        actions = np.load(os.path.join(ep_path, 'actions.npy'), allow_pickle=True)
        if actions.dtype == object:
            actions = np.array(actions.tolist(), dtype=np.float32)
        if np.any(np.isnan(actions)) or np.any(np.isinf(actions)): continue
        with open(os.path.join(ep_path, 'language.txt')) as f:
            language = f.read().strip()
        hf_data.append({'observation/image': imgs, 'actions': actions.tolist(), 'language_instruction': language})
    except: continue

random.seed(42)
random.shuffle(hf_data)
split_idx = int(0.9 * len(hf_data))
dataset_dict = DatasetDict({
    'train': Dataset.from_list(hf_data[:split_idx]),
    'validation': Dataset.from_list(hf_data[split_idx:]),
})
dataset_dict.save_to_disk(HF_DATASET_DIR)
print(f'Saved {len(hf_data)} episodes to {HF_DATASET_DIR} (train={split_idx}, val={len(hf_data)-split_idx})')

Converting to HF: 100%|██████████| 600/600 [00:00<00:00, 882.14it/s] 


Saving the dataset (0/1 shards):   0%|          | 0/540 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/60 [00:00<?, ? examples/s]

Saved 600 episodes to ./bridge_hf_synthetic (train=540, val=60)


## 6. Upload Dataset to S3

In [10]:
s3_client = boto3.client('s3')
local_root = Path(HF_DATASET_DIR)
files = [f for f in local_root.rglob('*') if f.is_file()]
print(f'Uploading {len(files)} files to s3://{bucket_name}/{S3_PREFIX}/')
for f in tqdm(files, desc='Uploading to S3'):
    relative = f.relative_to(local_root)
    s3_client.upload_file(str(f), bucket_name, f'{S3_PREFIX}/{relative}')
print(f'Dataset uploaded to: s3://{bucket_name}/{S3_PREFIX}/')
print('You can now run 02_training_job.ipynb')

Uploading 7 files to s3://sagemaker-us-east-1-783764584149/openvla-finetuning/datasets/bridge_hf_synthetic/


Uploading to S3: 100%|██████████| 7/7 [00:01<00:00,  4.97it/s]

Dataset uploaded to: s3://sagemaker-us-east-1-783764584149/openvla-finetuning/datasets/bridge_hf_synthetic/
You can now run 02_training_job.ipynb
